In [41]:
import logging
import pandas as pd
import numpy as np
import sklearn
from typing import Tuple
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing
from numpy.typing import NDArray
from sklearn.preprocessing import StandardScaler

In [42]:
logging.basicConfig(
    level=logging.INFO,
    format="[+%(relativeCreated)dms] %(levelname)s: %(message)s"
)

logger = logging.getLogger(__name__)
log = logger.info

In [43]:
housing = fetch_california_housing(download_if_missing=True, as_frame=False)

df = pd.DataFrame(data=housing.data, columns=housing.feature_names)

In [44]:
class GradientCheckError(Exception):
    """If propagation math fucks up then,"""
    pass

class DivergenceError(Exception):
    """Learing Rate too high, or Nan values occured"""
    pass

In [45]:

def data_processing_pipeline(X_raw: NDArray[np.float64], y_raw: NDArray[np.float64], seed: int = 42) -> Tuple[NDArray[np.float64], NDArray[np.float64],  NDArray[np.float64], NDArray[np.float64],  NDArray[np.float64], NDArray[np.float64]]:
    # reshape, because dot multiplication needs a 2d array, y contians 1d
    y_reshaped = y_raw.reshape(-1, 1)
    # first split in 70 percent, rest we use for val and test.
    X_train, X_temp, y_train, y_temp = train_test_split(
        X_raw,
        y_reshaped, 
        random_state=seed,
        test_size=0.3
    )
    # 2nd split between val and test, so 50 - 50 [100 - 70 = 30/2 = 15 - 15]
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=0.5,
        random_state=seed
    )

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    # WE DO NOT fit_transform on x_test, and x_val, just trasnform to prevent leakages.
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    # log info of all the data
    log(f"Data Split executed succesfully using random seed{seed}")
    log(f"Training metrics shape X: {X_train_scaled.shape}, y: {y_train.shape}")
    log(f"Validation metrics shape X: {X_val_scaled.shape}, y: {y_val.shape}")
    log(f"Testing metrics shape X: {X_test_scaled.shape}, y: {y_test.shape}")

    return X_train, X_val, X_test, y_train, y_val, y_test

In [46]:
class DenseLayer:
    def __init__(self, in_features: int, out_features: int, activation_str: str, rng: np.random.Generator):
        self.activation_str = activation_str

        if self.activation_str == "relu":
            std = np.sqrt(2.0/in_features)
        else:
            std = np.sqrt(1.0/in_features)

        self.weight = rng.standard_normal(size=(in_features, out_features)) * std
        self.bias = np.zeros((1, out_features))

    def forward(self, A_prev: NDArray[np.float64]) -> NDArray[np.float64]:
        self.A_prev = A_prev
        self Z =  A_prev @ self.weight  + self.bias
        return ACTIVATIONS[self.activation_str](self.Z)

    def backward(self, dA: NDArray[np.float64]) -> NDArray[np.float64]:
        

In [48]:
df

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25
...,...,...,...,...,...,...,...,...
20635,1.5603,25.0,5.045455,1.133333,845.0,2.560606,39.48,-121.09
20636,2.5568,18.0,6.114035,1.315789,356.0,3.122807,39.49,-121.21
20637,1.7000,17.0,5.205543,1.120092,1007.0,2.325635,39.43,-121.22
20638,1.8672,18.0,5.329513,1.171920,741.0,2.123209,39.43,-121.32
